# 🌳 Manacá Lab

### Experimentação com IA Brasileira

**Desenvolvido por Lucas Martins**  
Bibliotecário e Advogado  
IA, Ciência da Informação e Informação Jurídica

---

## Como testar

1. No menu do Colab, selecione **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**.
2. Depois clique em **Ambiente de execução → Executar tudo**.
3. Aguarde o carregamento do modelo.
4. Ao final, abra a interface do **Manacá Lab** e faça suas perguntas.

> As células técnicas estão recolhidas para facilitar a demonstração. O código continua disponível para fins de transparência e reprodutibilidade.

> **Aviso:** o Manacá-1B-Instruct é um modelo experimental de pesquisa. Verifique informações importantes em fontes confiáveis.

In [ ]:
!pip install -q -U transformers accelerate sentencepiece gradio


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_ID = 'menezesbruno/manaca-1b-instruct'

dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16 if torch.cuda.is_available()
    else torch.float32
)

tokenizer = AutoTokenizer.from_pretrained(MODELO_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODELO_ID,
    torch_dtype=dtype,
    device_map='auto'
)
model.eval()

print('✅ Manacá-1B-Instruct carregado com sucesso!')


---

## 💬 Interface de teste

Depois que o modelo carregar, use a interface abaixo ou abra o link público gerado pelo Gradio.


In [ ]:
import gradio as gr

def responder(instrucao, max_tokens, temperatura):
    if not instrucao or not instrucao.strip():
        return 'Digite uma instrução ou pergunta.'

    preambulo = (
        'Abaixo está uma instrução que descreve uma tarefa. '
        'Escreva uma resposta que atenda adequadamente ao pedido.'
    )
    prompt = f'{preambulo}\n\n### Instrução:\n{instrucao}\n\n### Resposta:\n'

    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=int(max_tokens),
            do_sample=True,
            temperature=float(temperatura),
            top_p=0.9,
            repetition_penalty=1.08,
            pad_token_id=tokenizer.eos_token_id
        )

    resposta_tokens = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(resposta_tokens, skip_special_tokens=True).strip()

with gr.Blocks(title='Manacá Lab — Instruct') as app:
    gr.Markdown('''
    # 🌳 Manacá Lab
    ### Experimentação com IA Brasileira

    Interface experimental com o **Manacá-1B-Instruct**.

    **Desenvolvido por Lucas Martins**  
    Bibliotecário e Advogado  
    IA, Ciência da Informação e Informação Jurídica

    ---
    ''')

    instrucao = gr.Textbox(
        label='Digite uma pergunta ou instrução',
        placeholder='Ex.: Explique o que é recuperação da informação jurídica.',
        lines=6
    )

    with gr.Row():
        max_tokens = gr.Slider(64, 512, value=256, step=32, label='Tamanho máximo da resposta')
        temperatura = gr.Slider(0.1, 1.2, value=0.3, step=0.1, label='Criatividade')

    botao = gr.Button('Gerar resposta', variant='primary')
    resultado = gr.Textbox(label='Resposta do Manacá', lines=14)

    gr.Examples(
        examples=[
            ['Explique em poucas linhas o que é recuperação da informação jurídica.'],
            ['Quais são três usos possíveis de IA em bibliotecas universitárias?'],
            ['Resuma a diferença entre legislação, jurisprudência e doutrina.'],
            ['Explique o que é RAG em linguagem simples.']
        ],
        inputs=instrucao
    )

    gr.Markdown('''
    ---
    **Aviso:** modelo experimental de pesquisa. Verifique informações importantes em fontes confiáveis.
    ''')

    botao.click(
        fn=responder,
        inputs=[instrucao, max_tokens, temperatura],
        outputs=resultado
    )

app.launch(share=True, inline=False)
